# M14 · Les maths des ensembles — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookMaths/blob/main/chapitres/m14_ensembles/03_notebook.ipynb)

La partie 0 vérifie tes exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wbm.check`). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/m14_ensembles/03_notebook.ipynb`, créée par `python tools/start_chapter.py M14`) ; la version du dépôt est mise à jour par Claude. Sur Colab, le badge ouvre la version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | M14.1 à M14.5, M14.9 | Les exercices papier (vérification) | ✏️ 🔗 | | |
| A | M14.11 | La borne de Chernoff et la variance d'une moyenne pondérée · `chernoff_bound, variance_of_average` | 🔨 | ★ | 20 |
|  | M14.12 | Les poids de l'algorithme Hedge · `hedge_weights` | 🔨 | ★ | 15 |
| B | M14.13 | Simuler la moyenne de deux groupes de modèles corrélés, et retrouver aᵀΣa et son plancher | 🔬 | ★★ | 20 |
|  | M14.14 | Les poids exponentiels au fil des tours : quel expert domine, selon le learning rate | 🎨 | ★ | 15 |
| C | M14.15 | Chasse à l'erreur : P(X ≥ k) calculée comme P(X > k), des covariances oubliées, des poids non renormalisés | 🐛 | ★ | 15 |
|  | M14.16 | Un test qui vérifie que la borne de Chernoff majore la queue exacte de `scipy.stats`, sur une grille de n, p et a | 🛠️ | ★ | 15 |
|  | M14.17 | Défi : suivre le meilleur de dix experts avec Hedge sans connaître l'horizon, un regret sous la borne théorique sur 1 000 tours | 🏆 | ★★★ | 25 |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (a few seconds per cell); False = bigger experiments
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookMaths.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookMaths")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WBM_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wbm").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wbm").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier de workbookMaths.")

os.environ["WBM_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wbm  # noqa: E402

cfg = wbm.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M14")  # your own library

## Objectifs et rappel express

- Vérifier tes calculs papier (partie 0) : la queue d'une loi binomiale et ses trois bornes, la variance d'une moyenne pondérée et les meilleurs poids, la limite $(1 + x/n)^n \to e^x$, les poids de Hedge et son regret, le gradient d'une loss par rapport aux prédictions, le plancher de variance de deux groupes de modèles.
- Écrire les trois fonctions de `mymath/ensembles.py` et les tester.
- Simuler un plancher de variance, tracer les poids de Hedge, chasser des erreurs, tester une borne, et relever un défi : Hedge sans connaître l'horizon.

**Rappel express.** $P(X \ge k) = \sum_{j \ge k}\binom{n}{j}p^j(1 - p)^{n-j}$ = `binom.sf(k - 1, n, p)` ; pour $a > p$, $P(X \ge na) \le e^{-n\,\mathrm{KL}(a \Vert p)}$ (Chernoff), avec $\mathrm{KL}(a \Vert p) = a\ln\frac{a}{p} + (1 - a)\ln\frac{1 - a}{1 - p} \ge 2(a - p)^2$ (Hoeffding : $e^{-2n(a - p)^2}$) ; $\mathbb{V}\big(\sum_i w_iX_i\big) = \mathbf{w}^\top\boldsymbol{\Sigma}\mathbf{w}$, meilleurs poids $\propto \boldsymbol{\Sigma}^{-1}\mathbf{1}$, variance minimale $\frac{1}{\mathbf{1}^\top\boldsymbol{\Sigma}^{-1}\mathbf{1}}$ ; une part commune laisse un plancher ; $(1 + x/n)^n \to e^x$, erreur relative $\approx -\frac{x^2}{2n}$ ; Hedge : $w_t[k] \propto e^{-\eta L_t[k]}$ (retrancher $\min_j L_t[j]$), $R_T \le \frac{\ln K}{\eta} + \frac{\eta T}{8}$, $\eta = \sqrt{8\ln K/T}$ donne $R_T \le \sqrt{\frac{T\ln K}{2}}$ ; $\frac{\partial}{\partial F}e^{-yF} = -y\,e^{-yF}$, et pour Huber $-\operatorname{clip}(y - F, -\delta, \delta)$.

## Partie 0 · Vérifier tes exercices papier (✏️ M14.1 à M14.5, 🔗 M14.9)

Fais d'abord les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse ici : **la valeur** que tu as trouvée, sous la forme indiquée en commentaire, jamais le calcul qui y mène :
- un entier, ou un nombre avec un point décimal, `-1.25`, arrondi comme le demande la question ;
- une fraction irréductible entre guillemets, `"5/9"` ;
- une liste `[0.15, -2]` ; dans une liste, une fraction s'écrit entre guillemets, `["1/4", "3/4"]`, ou comme un calcul, sans guillemets : `[1/4, 3/4]`.

Les réponses pas encore remplies affichent ⏳.

In [ ]:
from fractions import Fraction   # Fraction(2, 9) is accepted too, as "2/9"

**Ex M14.1 — La fiabilité d'un système « k parmi n » : au moins k composants qui marchent**

In [ ]:
# Ex M14.1: the values you found on paper
answer_M14_1a = ...  # a) a number (4 decimals)
answer_M14_1b = ...  # b) a number (4 decimals)
answer_M14_1c = ...  # c) a number
answer_M14_1d = ...  # d) a number (4 decimals)
answer_M14_1e = ...  # e) a number (4 decimals)
answer_M14_1f = ...  # f) a number (4 decimals)
answer_M14_1g = ...  # g) a number (4 decimals)
answer_M14_1h = ...  # h) a number (4 decimals)

for letter, answer in zip("abcdefgh", [answer_M14_1a, answer_M14_1b, answer_M14_1c, answer_M14_1d, answer_M14_1e, answer_M14_1f, answer_M14_1g, answer_M14_1h]):
    wbm.check(f"M14.1{letter}", answer)

**Ex M14.2 — La variance de la moyenne de trois mesures corrélées, par la matrice de covariance**

In [ ]:
# Ex M14.2: the values you found on paper
answer_M14_2a = ...  # a) an irreducible fraction, as text
answer_M14_2b = ...  # b) an irreducible fraction, as text
answer_M14_2c = ...  # c) an irreducible fraction, as text
answer_M14_2d = ...  # d) a list of 3 fractions, as texts (["2/7", "5/7"]) or calculations, or with 6 decimals
answer_M14_2e = ...  # e) an irreducible fraction, as text
answer_M14_2f = ...  # f) a number (4 decimals)

for letter, answer in zip("abcdef", [answer_M14_2a, answer_M14_2b, answer_M14_2c, answer_M14_2d, answer_M14_2e, answer_M14_2f]):
    wbm.check(f"M14.2{letter}", answer)

**Ex M14.3 — (1 + x/n)ⁿ pour quelques n, et des intérêts composés en continu**

In [ ]:
# Ex M14.3: the values you found on paper
answer_M14_3a = ...  # a) a number (6 decimals)
answer_M14_3b = ...  # b) a number (6 decimals)
answer_M14_3c = ...  # c) a number (6 decimals)
answer_M14_3d = ...  # d) a number (6 decimals)
answer_M14_3e = ...  # e) a number (6 decimals)
answer_M14_3f = ...  # f) a number (6 decimals)
answer_M14_3g = ...  # g) a number (3 decimals)
answer_M14_3h = ...  # h) an integer

for letter, answer in zip("abcdefgh", [answer_M14_3a, answer_M14_3b, answer_M14_3c, answer_M14_3d, answer_M14_3e, answer_M14_3f, answer_M14_3g, answer_M14_3h]):
    wbm.check(f"M14.3{letter}", answer)

**Ex M14.4 — Les poids exponentiels de cinq experts après trois tours**

In [ ]:
# Ex M14.4: the values you found on paper
answer_M14_4a = ...  # a) a list of 5 fractions, as texts (["2/7", "5/7"]) or calculations, or with 6 decimals
answer_M14_4b = ...  # b) a list of 5 fractions, as texts (["2/7", "5/7"]) or calculations, or with 6 decimals
answer_M14_4c = ...  # c) a list of 5 fractions, as texts (["2/7", "5/7"]) or calculations, or with 6 decimals
answer_M14_4d = ...  # d) an irreducible fraction, as text
answer_M14_4e = ...  # e) an integer
answer_M14_4f = ...  # f) a number (4 decimals)
answer_M14_4g = ...  # g) a number (3 decimals)
answer_M14_4h = ...  # h) a number (4 decimals)

for letter, answer in zip("abcdefgh", [answer_M14_4a, answer_M14_4b, answer_M14_4c, answer_M14_4d, answer_M14_4e, answer_M14_4f, answer_M14_4g, answer_M14_4h]):
    wbm.check(f"M14.4{letter}", answer)

**Ex M14.5 — Le gradient de la loss exponentielle e^(−yF) et de la loss de Huber par rapport aux prédictions, et les poids exponentiels qu'il fait apparaître**

In [ ]:
# Ex M14.5: the values you found on paper
answer_M14_5a = ...  # a) a list of 5 numbers
answer_M14_5b = ...  # b) a number (3 decimals)
answer_M14_5c = ...  # c) a list of 5 numbers (3 decimals)
answer_M14_5d = ...  # d) a list of 5 numbers (4 decimals)
answer_M14_5e = ...  # e) an integer
answer_M14_5f = ...  # f) a list of 4 numbers
answer_M14_5g = ...  # g) a number (4 decimals)

for letter, answer in zip("abcdefg", [answer_M14_5a, answer_M14_5b, answer_M14_5c, answer_M14_5d, answer_M14_5e, answer_M14_5f, answer_M14_5g]):
    wbm.check(f"M14.5{letter}", answer)

**Ex M14.9 — Pont : ajouter des modèles très corrélés ne sert presque à rien, un plancher de variance sur deux groupes de modèles**

In [ ]:
# Ex M14.9: the values you found on paper
answer_M14_9a = ...  # a) an irreducible fraction, as text
answer_M14_9b = ...  # b) an irreducible fraction, as text
answer_M14_9c = ...  # c) an irreducible fraction, as text
answer_M14_9d = ...  # d) an irreducible fraction, as text
answer_M14_9e = ...  # e) an irreducible fraction, as text
answer_M14_9f = ...  # f) an irreducible fraction, as text
answer_M14_9g = ...  # g) an irreducible fraction, as text
answer_M14_9h = ...  # h) an irreducible fraction, as text

for letter, answer in zip("abcdefgh", [answer_M14_9a, answer_M14_9b, answer_M14_9c, answer_M14_9d, answer_M14_9e, answer_M14_9f, answer_M14_9g, answer_M14_9h]):
    wbm.check(f"M14.9{letter}", answer)

## Partie A · Ta librairie : la borne de Chernoff, la variance d'une moyenne et les poids de Hedge

Fiche §1, §2 et §4. Les trois fonctions de `mymath/ensembles.py`, et leurs tests.

In [ ]:
# Tools of the notebook (parts A to C)
import ast
import inspect
import io
import linecache
import math
import re
import textwrap
import time
import tokenize
import warnings
from fractions import Fraction

import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import binom

from wbm.nbtools import StopCheck, checked, drawn, filled, fr, own_code, run_pytest_group, verdict


def raised(function, *args, **kwargs):
    """The name of the exception that function(*args, **kwargs) raises, or "aucune" if it returns.

    NotImplementedError (a TODO not done yet) goes through, for wbm.attempt to show ⏳; so do the errors of a code
    that cannot run (a missing import, a misspelt name such as ValueErorr), for the check to name them."""
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            function(*args, **kwargs)
    except (NotImplementedError, NameError, AttributeError, ImportError):
        raise
    except Exception as exc:  # noqa: BLE001 - any other error is the answer
        return type(exc).__name__
    return "aucune"

### Ex M14.11 — La borne de Chernoff et la variance d'une moyenne pondérée · `chernoff_bound, variance_of_average` 🔨 ★ ⏱️ 20 min
**Objectif :** programmer la borne de Chernoff d'une queue binomiale et la variance d'une moyenne pondérée de variables corrélées, et vérifier que la borne majore la queue exacte.  
**Prérequis :** fiche §1 et §2 · **mymath :** `ensembles.py` · **Parcours :** C, E

> **Mode d'emploi `mymath`** (comme en M03). Ouvre ta copie `mon_travail/mymath/ensembles.py`, lis la docstring du module puis celle de la fonction, remplace la ligne `raise NotImplementedError(...)` par ton code, **enregistre** le fichier, puis exécute les cellules ci-dessous : la première recharge ta librairie. NumPy et le module `math` sont permis ; SciPy et scikit-learn sont interdits (les tests comparent tes résultats à ceux de `scipy.stats` et `scipy.special`, et un test vérifie que tu ne t'en sers pas). Ne modifie jamais les tableaux reçus : travaille sur une copie, `np.array(x, dtype=float)`. Les nombres se rendent en `float` de Python, les vecteurs et les matrices en tableaux NumPy de flottants.

Écris, en suivant les docstrings (fiche §1 et §2) :
- `chernoff_bound(n, p, a)` : `ValueError` si $n$ n'est pas un entier $\ge 0$ (un booléen, `True`, n'est pas un nombre d'essais), ou si $p$ ou $a$ n'est pas un nombre de $[0, 1]$ (`float(\"nan\")` compris) ; `1.0` si $a \le p$ (la borne ne dit rien) ; `0.0` si $p = 0 < a$ (l'événement est impossible) ; sinon $e^{-n\,\mathrm{KL}(a \Vert p)}$, avec $\mathrm{KL}(a \Vert p) = a\ln\frac{a}{p} + (1 - a)\ln\frac{1 - a}{1 - p}$ en nats, où $0\ln 0$ compte pour 0 (le cas $a = 1$) ;
- `variance_of_average(cov, weights=None)` : `ValueError` si `cov` n'est pas une matrice carrée symétrique, si les poids n'ont pas la bonne longueur, ont une valeur négative ou une somme différente de 1, ou si le résultat est négatif au-delà de l'arrondi ; sinon $\mathbf{w}^\top\boldsymbol{\Sigma}\mathbf{w}$, avec les poids $\frac{1}{M}$ quand `weights` est omis.

La vérification calcule, avec les données de la cellule fournie :
- a) `chernoff_bound(32, 0.24, 0.45)`, la borne de $P(X \ge 14{,}4)$ pour $X$ de loi $\mathcal{B}(32 ;\ 0{,}24)$ (6 décimales) ;
- b) `chernoff_bound(32, 0.24, 0.15)`, un cas où $a \le p$ ;
- c) `variance_of_average(COV_11)`, la variance de la moyenne simple de quatre variables de matrice de covariance `COV_11` (6 décimales) ;
- d) `variance_of_average(COV_11, weights=W_11)` (6 décimales) ;
- e) vrai ou faux : sur la grille de la cellule fournie (4 valeurs de $n$, 4 de $p$, 21 de $a$), ta borne est toujours au-dessus de la queue exacte $P(X \ge na)$, calculée par `binom.sf(math.ceil(n * a - 1e-9) - 1, n, p)` de `scipy.stats` (sinon, la vérification affiche le premier cas où elle passe dessous) ;
- f) le nom de l'erreur levée par `variance_of_average(COV_11, weights=[0.2, 0.3, 0.3, 0.3])`.

Dans ta copie : calcule $\mathrm{KL}(0{,}45 \Vert 0{,}24)$ à la main, puis a), et compare-la à la borne de Hoeffding $e^{-2n(a - p)^2}$ et à la queue exacte $P(X \ge 15)$. Que donnerait c) sans les covariances, et pourquoi c) est-il plus grand ? La grille de e) contient `a = np.linspace(0, 1, 21)[12]`, qui vaut `0.6000000000000001` : que vaut `25 * a`, et quelle queue calculerait `math.ceil(n * a) - 1` sans le `- 1e-9` ? Pourquoi f) est-il refusé plutôt que renormalisé en silence ?

In [ ]:
N_11, P_11, A_11 = 32, 0.24, 0.45         # X of law B(32 ; 0.24), and the event X ≥ 32 × 0.45 = 14.4
COV_11 = np.array([[2.5, 0.8, -0.4, 1.1],
                   [0.8, 3.2, 0.6, -0.7],
                   [-0.4, 0.6, 1.8, 0.3],
                   [1.1, -0.7, 0.3, 2.9]])        # the covariance matrix of four variables
W_11 = np.array([0.1, 0.2, 0.45, 0.25])          # weights of sum 1
GRID_N_11, GRID_P_11, GRID_A_11 = (8, 25, 70, 160), (0.15, 0.4, 0.68, 0.9), np.linspace(0.0, 1.0, 21)

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M14")  # reload your saved file

In [ ]:
with checked("M14.11a"):
    wbm.check("M14.11a", mymath.ensembles.chernoff_bound(N_11, P_11, A_11), computed=True)
with checked("M14.11b"):
    wbm.check("M14.11b", mymath.ensembles.chernoff_bound(N_11, P_11, 0.15), computed=True)
with checked("M14.11c"):
    wbm.check("M14.11c", mymath.ensembles.variance_of_average(COV_11), computed=True)
with checked("M14.11d"):
    wbm.check("M14.11d", mymath.ensembles.variance_of_average(COV_11, weights=W_11), computed=True)
with checked("M14.11e"):
    below_11 = None                              # the first (n, p, a) where your bound is below the exact tail
    for n, p, a in ((n, p, float(a)) for n in GRID_N_11 for p in GRID_P_11 for a in GRID_A_11):
        bound_11, tail_11 = mymath.ensembles.chernoff_bound(n, p, a), float(binom.sf(math.ceil(n * a - 1e-9) - 1, n, p))
        if not bound_11 >= tail_11 - 1e-12:
            below_11 = (n, p, a, bound_11, tail_11)
            break
    wbm.check("M14.11e", below_11 is None, computed=True)
    if below_11 is not None:
        print(f"   premier cas : n = {below_11[0]}, p = {fr(below_11[1], 2)}, a = {fr(below_11[2], 2)} : ta borne "
              f"vaut {below_11[3]:.4g}, la queue exacte {below_11[4]:.4g}".replace(".", ","))
with checked("M14.11f"):
    wbm.check("M14.11f", raised(mymath.ensembles.variance_of_average, COV_11, [0.2, 0.3, 0.3, 0.3]), computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m14_ensembles.py", "test_chernoff or test_variance or test_ensembles_uses")

### Ex M14.12 — Les poids de l'algorithme Hedge · `hedge_weights` 🔨 ★ ⏱️ 15 min
**Objectif :** programmer les poids exponentiels de Hedge sans débordement, et calculer la loss de l'algorithme et son regret.  
**Prérequis :** fiche §4 · **mymath :** `ensembles.py` · **Parcours :** C

Écris `hedge_weights(losses, eta)` (fiche §4), en suivant sa docstring : `ValueError` si `losses` n'est pas un tableau 2-D d'au moins une colonne, ou si $\eta \le 0$ ; sinon le tableau $(T + 1) \times K$ des poids. Sa ligne $t$ (de 0 à $T$) donne les poids avant le tour $t + 1$ : ils sont proportionnels à $e^{-\eta L[k]}$, où $L[k]$ est la loss cumulée de l'expert $k$ sur les $t$ premiers tours ; la ligne 0, avant toute loss, est uniforme, et la dernière vient après le dernier tour. Retranche la plus petite loss cumulée de chaque ligne avant l'exponentielle, puis normalise chaque ligne.

La cellule fournie donne les losses de quatre experts sur six tours, `LOSSES_12`, et $\eta = 1{,}5$. La vérification calcule :
- a) la dernière ligne de `hedge_weights(LOSSES_12, 1.5)`, les poids après le sixième tour (une liste, 6 décimales) ;
- b) vrai ou faux : chaque ligne somme à 1, et la première est uniforme ;
- c) vrai ou faux : avec des losses de l'ordre de $10^4$ (`BIG_12`, les mêmes multipliées par 20 000), les poids sont finis et chaque ligne somme à 1 ;
- d) la loss totale de l'algorithme, $\sum_t \mathbf{w}_t \cdot \boldsymbol{\ell}_t$, avec tes poids (6 décimales) ;
- e) son regret, d) moins la loss totale du meilleur expert (6 décimales) ;
- f) le nom de l'erreur levée par `hedge_weights(LOSSES_12, 0.0)`.

Dans ta copie : calcule les losses cumulées des quatre experts après six tours, et retrouve a) avec une calculatrice. Quel expert a le plus de poids à la fin, et combien de tours a-t-il gagnés (la plus petite loss du tour) ? Pour c), que donnerait $e^{-\eta L}$ sans retrancher le minimum, et pourquoi la soustraction ne change-t-elle pas les poids ? Compare e) à la borne $\frac{\ln K}{\eta} + \frac{\eta T}{8}$.

In [ ]:
LOSSES_12 = np.array([[0.3, 0.8, 0.5, 0.1],
                      [0.6, 0.2, 0.4, 0.9],
                      [0.1, 0.7, 0.3, 0.6],
                      [0.9, 0.1, 0.2, 0.5],
                      [0.4, 0.5, 0.6, 0.2],
                      [0.2, 0.3, 0.1, 0.7]])    # the losses of 4 experts (columns) over 6 rounds (rows)
ETA_12 = 1.5
BIG_12 = 20000.0 * LOSSES_12                  # the same losses, of the order of 10⁴

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M14")  # reload your saved file

In [ ]:
shape_told_12 = []                               # the error of shape is said once, not by every block


def learner_weights_12(losses):
    """Your weights for LOSSES_12-like losses, after a check of their shape (one row more than the rounds)."""
    w = np.asarray(mymath.ensembles.hedge_weights(losses, ETA_12), dtype=float)
    if w.shape != (len(losses) + 1, 4):
        if not shape_told_12:
            print(f"❌ Ex M14.12 : hedge_weights doit rendre un tableau ({len(losses) + 1}, 4) : une ligne avant chaque "
                  f"tour, puis une après le dernier ; reçu la forme {w.shape}")
            shape_told_12.append(True)
        raise StopCheck
    return w


with checked("M14.12a"):
    wbm.check("M14.12a", np.asarray(mymath.ensembles.hedge_weights(LOSSES_12, ETA_12))[-1].tolist(), computed=True)
with checked("M14.12b"):
    w_12 = learner_weights_12(LOSSES_12)
    rows_12 = bool(np.allclose(w_12.sum(axis=1), 1.0, rtol=0, atol=1e-12))
    first_12 = bool(np.allclose(w_12[0], w_12[0].mean(), rtol=0, atol=1e-12))     # the same weight for every expert
    wbm.check("M14.12b", rows_12 and first_12 and bool(np.allclose(w_12[0], 0.25)), computed=True)
    if not rows_12:
        print("   des lignes ne somment pas à 1 : normalise CHAQUE ligne, w / w.sum(axis=1, keepdims=True)")
    if not first_12:
        print("   la première ligne vient avant toute loss : elle doit être uniforme")
with checked("M14.12c"):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        big_12 = learner_weights_12(BIG_12)
    finite_12 = bool(np.all(np.isfinite(big_12)))
    rows_big_12 = finite_12 and bool(np.allclose(big_12.sum(axis=1), 1.0))
    wbm.check("M14.12c", rows_big_12, computed=True)
    if not finite_12:
        print("   des poids non finis : retranche la plus petite loss cumulée de chaque ligne avant l'exponentielle")
    elif not rows_big_12:
        print("   des poids finis, mais des lignes qui ne somment pas à 1 : normalise CHAQUE ligne, "
              "w / w.sum(axis=1, keepdims=True)")
with checked("M14.12d"):
    w_12 = learner_weights_12(LOSSES_12)
    wbm.check("M14.12d", float(np.sum(w_12[:-1] * LOSSES_12)), computed=True)
with checked("M14.12e"):
    w_12 = learner_weights_12(LOSSES_12)
    wbm.check("M14.12e", float(np.sum(w_12[:-1] * LOSSES_12) - LOSSES_12.sum(axis=0).min()), computed=True)
with checked("M14.12f"):
    wbm.check("M14.12f", raised(mymath.ensembles.hedge_weights, LOSSES_12, 0.0), computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m14_ensembles.py", "test_hedge or test_ensembles_uses")

## Partie B · Une simulation et une figure

Fiche §2 et §4. Le plancher de variance de deux groupes de modèles corrélés, retrouvé par simulation, puis les poids de Hedge au fil des tours.

### Ex M14.13 — Simuler la moyenne de deux groupes de modèles corrélés, et retrouver aᵀΣa et son plancher 🔬 ★★ ⏱️ 20 min
**Objectif :** simuler la moyenne de modèles dont les erreurs ont une part commune par groupe, retrouver sa variance par $\mathbf{a}^\top\boldsymbol{\Sigma}\mathbf{a}$, et mesurer le plancher que laissent les covariances.  
**Prérequis :** fiche §2 ; Ex M14.9 et M14.11 · **Parcours :** C, E

On moyenne $2m$ modèles de régression de deux groupes, comme en 🔗 M14.9, mais avec d'autres nombres. Pour un exemple donné, l'erreur d'un modèle du groupe A vaut $Z_A + E_i$, celle d'un modèle du groupe B, $Z_B + E_i$ : les parts communes $(Z_A, Z_B)$ suivent une loi normale centrée de matrice de covariance `COV_Z_13` $= \begin{pmatrix} 0{,}6 & 0{,}5 \\ 0{,}5 & 1{,}4 \end{pmatrix}$, et les parts propres $E_i$ sont normales, centrées, de variance $2{,}2$, indépendantes entre elles et de $(Z_A, Z_B)$. Écris :
- `simulate_mean_variance(m, n_draws, rng)` : `n_draws` tirages ; chacun tire **une** paire $(Z_A, Z_B)$ (`rng.multivariate_normal`) et les $2m$ parts propres (`rng.normal`), forme les erreurs des $m$ modèles du groupe A puis des $m$ modèles du groupe B, et en fait la moyenne ; la fonction rend la variance empirique (`np.var`) des `n_draws` moyennes, un `float` ;
- `models_covariance(m)` : la matrice de covariance $2m \times 2m$ des erreurs des $2m$ modèles ($m$ du groupe A, puis $m$ du groupe B).

Une cellule les appelle pour $m$ = 1, 2, 4, …, 64, avec 40 000 tirages et un générateur de graine 4331, et passe chaque matrice à `variance_of_average` (celle de la référence, `ref_13.variance_of_average`, ou la tienne) : la moyenne simple des $2m$ modèles a la variance $\mathbf{a}^\top\boldsymbol{\Sigma}\mathbf{a}$, avec $\mathbf{a} = \left(\frac{1}{2m}, \ldots, \frac{1}{2m}\right)$. Calcule ensuite :
- a) la variance théorique de la moyenne pour $m = 6$ (6 décimales) ;
- b) le plancher, la limite de cette variance quand $m \to \infty$ (6 décimales, ou une fraction entre guillemets) ;
- c) vrai ou faux : pour chaque $m$ de la liste, la variance simulée est à moins de 5 % de la variance théorique ;
- d) pour $m = 32$, la variance de la moyenne **pondérée** qui donne au groupe A le poids total $\frac{3}{4}$ (chaque modèle du groupe A le poids $\frac{3}{128}$, chaque modèle du groupe B le poids $\frac{1}{128}$), calculée par `variance_of_average(models_covariance(32), weights=...)` (4 décimales).

Une cellule trace ensuite les deux variances en fonction de $m$, avec ton plancher.

Dans ta copie : écris la moyenne des $2m$ erreurs comme une part commune plus une moyenne de parts propres, et retrouve a) et b) sans `variance_of_average`. Pourquoi la variance simulée ne tombe-t-elle jamais exactement sur la variance théorique, et de combien s'en écarte-t-elle typiquement avec 40 000 tirages (l'écart-type relatif d'une variance empirique est d'environ $\sqrt{2/n}$) ? Pourquoi la moyenne pondérée de d) fait-elle mieux que la moyenne simple des mêmes 64 modèles ? Quelle proportion $\alpha$ du groupe A donnerait le plus petit plancher ? Quelle est la corrélation entre deux modèles du groupe B ? Que changerait une covariance $\operatorname{Cov}(Z_A, Z_B)$ négative ?

In [ ]:
VAR_A_13, VAR_B_13, COV_AB_13 = 0.6, 1.4, 0.5      # the common parts: V(Z_A), V(Z_B) and Cov(Z_A, Z_B)
COV_Z_13 = np.array([[VAR_A_13, COV_AB_13], [COV_AB_13, VAR_B_13]])
OWN_13 = 2.2                                          # the variance of each own part E_i
GROUP_SIZES_13, N_DRAWS_13 = [1, 2, 4, 8, 16, 32, 64], 40000   # the numbers m of models per group, and the draws
ref_13 = wbm.load_mymath("ref", alias="mymath_ref").ensembles

In [ ]:
def simulate_mean_variance(m, n_draws, rng):
    """The empirical variance of the mean of the 2m models (m of group A, then m of group B) over n_draws draws: each
    draw takes one pair (Z_A, Z_B) of covariance COV_Z_13 and 2m independent own parts of variance OWN_13, all with
    the generator rng."""
    raise NotImplementedError("simulate_mean_variance n'est pas encore écrite (04_indices.md si tu bloques)")


def models_covariance(m):
    """The (2m, 2m) covariance matrix of the errors of the 2m models: m of group A, then m of group B."""
    raise NotImplementedError("models_covariance n'est pas encore écrite (04_indices.md si tu bloques)")


answer_M14_13a = answer_M14_13b = answer_M14_13c = answer_M14_13d = ...
with wbm.attempt("M14.13"):          # your two functions are called here: ⏳ while they are not written
    rng_13 = np.random.default_rng(4331)
    simulated_13 = [simulate_mean_variance(m, N_DRAWS_13, rng_13) for m in GROUP_SIZES_13]
    theoretical_13 = [ref_13.variance_of_average(models_covariance(m)) for m in GROUP_SIZES_13]
    answer_M14_13a = ...  # a) a number (6 decimals): the theoretical variance for m = 6
    answer_M14_13b = ...  # b) a number (6 decimals) or a fraction as text: the floor
    answer_M14_13c = ...  # c) True or False
    answer_M14_13d = ...  # d) a number (4 decimals): the weighted average for m = 32

In [ ]:
for letter, answer in zip("abcd", [answer_M14_13a, answer_M14_13b, answer_M14_13c, answer_M14_13d]):
    wbm.check(f"M14.13{letter}", answer)

with checked("M14.13 figure"):
    fig, ax = plt.subplots(figsize=(7.0, 3.8))
    try:
        rng_fig_13 = np.random.default_rng(4331)
        simulated_13 = [simulate_mean_variance(m, N_DRAWS_13, rng_fig_13) for m in GROUP_SIZES_13]
        theoretical_13 = [ref_13.variance_of_average(models_covariance(m)) for m in GROUP_SIZES_13]
        ax.plot(GROUP_SIZES_13, theoretical_13, "o-", color="C0", label="variance théorique (variance_of_average)")
        ax.plot(GROUP_SIZES_13, simulated_13, "s--", color="C1", label="variance simulée (40 000 tirages)")
        try:
            floor_13 = float(Fraction(str(answer_M14_13b).replace(" ", ""))) if filled(answer_M14_13b) else None
        except (ValueError, ZeroDivisionError, NameError):
            floor_13 = None
        if floor_13 is not None:
            ax.axhline(floor_13, color="C3", ls=":", label="ton plancher, b)")
        ax.set_xscale("log", base=2)
        ax.set_xlabel("m, le nombre de modèles de chaque groupe (échelle log)")
        ax.set_ylabel("variance de la moyenne des 2m modèles")
        ax.legend(fontsize=8)
    except BaseException:
        plt.close(fig)
        raise
    plt.show()

### Ex M14.14 — Les poids exponentiels au fil des tours : quel expert domine, selon le learning rate 🎨 ★ ⏱️ 15 min
**Objectif :** tracer les poids de Hedge au fil des tours pour plusieurs learning rates, et voir comment η règle la vitesse et les à-coups avec lesquels ils suivent le meilleur expert.  
**Prérequis :** fiche §4 ; Ex M14.12 · **Parcours :** C

Cinq experts, 80 tours : la cellule fournie tire leurs losses dans $[0, 1]$ (graine 4334). L'expert 3 est le meilleur en moyenne, mais il commence mal. Écris `draw_hedge(axes, losses, etas)`, qui, pour chaque learning rate de `etas`, dessine dans l'axe correspondant de `axes` (sans en ouvrir d'autre) :
- le poids de chaque expert au fil des tours, une courbe (`ax.plot`) ou une aire empilée (`ax.stackplot`) par expert, avec les $T + 1 = 81$ lignes de `hedge_weights(losses, eta)` (la tienne, `mymath.ensembles.hedge_weights`, ou celle de la référence, `ref_14.hedge_weights`) ;
- un titre qui donne la valeur de $\eta$, une légende et des noms d'axes.

La cellule de vérification appelle ta fonction avec trois axes, pour $\eta$ = 0,15, 0,6 et 3, contrôle que les dessins sont dans les axes reçus, qu'il y a une courbe (ou une aire) par expert, que les valeurs tracées sont les bons poids, et que chaque titre donne $\eta$, puis affiche ta figure.

Dans ta copie : pour chaque $\eta$, quel expert domine à la fin, et à partir de quel tour son poids dépasse-t-il 0,5 pour de bon ? Que font les poids pendant les vingt premiers tours avec $\eta = 3$, et pourquoi ces à-coups coûtent-ils au regret (fiche §4, le second terme de la borne) ? Avec $\eta = 0{,}15$, l'expert 3 a-t-il assez de poids à la fin ? Calcule le regret de chacun sur ces 80 tours (avec `hedge_weights`), puis sa garantie $\frac{\ln K}{\eta} + \frac{\eta T}{8}$ : lequel des trois choisirais-tu, et que dit la formule $\eta = \sqrt{8\ln K/T}$ ?

In [ ]:
rng_14 = np.random.default_rng(4334)
MEANS_14 = np.array([0.46, 0.52, 0.34, 0.6, 0.49])                       # the mean loss of each of the 5 experts
LOSSES_14 = np.clip(MEANS_14 + rng_14.normal(0.0, 0.2, size=(80, 5)), 0.0, 1.0)  # 80 rounds
LOSSES_14[:12, 2] = np.clip(LOSSES_14[:12, 2] + 0.3, 0.0, 1.0)            # expert 3 starts badly
LOSSES_14 = np.round(LOSSES_14, 2)
ETAS_14 = (0.15, 0.6, 3.0)
ref_14 = wbm.load_mymath("ref", alias="mymath_ref").ensembles

In [ ]:
def draw_hedge(axes, losses, etas):
    """In each axes received (one per learning rate of etas, and no other): the weight of each expert before each
    round and after the last one, from hedge_weights, with a title that gives eta."""
    raise NotImplementedError("draw_hedge n'est pas encore écrite (04_indices.md si tu bloques)")

In [ ]:
with checked("M14.14"):
    results_14 = []

    def report_14(ok, success, failure):
        results_14.append(bool(ok))
        print(("✅ " + success) if ok else ("❌ " + failure))

    def drawn_series_14(ax, rows):
        """The series of `rows` values drawn in ax: the y data of its lines, else the heights of its filled areas."""
        series = [np.asarray(line.get_ydata(), dtype=float) for line in ax.lines if len(line.get_ydata()) == rows]
        if series:
            return series
        for collection in ax.collections:
            for path in collection.get_paths():
                points = np.asarray(path.vertices, dtype=float)
                xs = np.unique(np.round(points[:, 0], 9))
                if len(xs) == rows:
                    series.append(np.array([np.ptp(points[np.isclose(points[:, 0], x), 1]) for x in xs]))
        return series

    def own_weights_right_14():
        """True or False: your hedge_weights (M14.12) gives the right weights on these losses; None if it cannot say
        (not written, or it fails): draw_hedge may use ref_14.hedge_weights instead."""
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", RuntimeWarning)
                own = [np.asarray(mymath.ensembles.hedge_weights(LOSSES_14, eta), dtype=float) for eta in ETAS_14]
        except Exception:  # noqa: BLE001 - NotImplementedError included: nothing to say about it here
            return None
        return all(o.shape == (len(LOSSES_14) + 1, 5) and np.allclose(o, ref_14.hedge_weights(LOSSES_14, eta),
                                                                       rtol=0, atol=1e-6)
                   for o, eta in zip(own, ETAS_14))

    fig, axes = plt.subplots(1, len(ETAS_14), figsize=(13.0, 3.8), sharey=True)
    try:
        opened_14 = drawn(draw_hedge, axes, LOSSES_14, ETAS_14)
    except BaseException:
        plt.close(fig)
        raise
    report_14(not opened_14 and all(ax.lines or ax.collections for ax in axes), "draw_hedge dessine dans les axes reçus",
              "draw_hedge doit dessiner dans chacun des axes reçus, sans ouvrir de figure (ni plt.figure ni plt.subplots)")
    founds_14 = []
    for ax, eta in zip(axes, ETAS_14):
        want = ref_14.hedge_weights(LOSSES_14, eta)
        series = drawn_series_14(ax, want.shape[0])
        founds_14.append(sum(any(np.allclose(s, want[:, k], rtol=0, atol=1e-6) for s in series)
                             for k in range(want.shape[1])))
    if any(found < 5 for found in founds_14) and own_weights_right_14() is False:
        report_14(False, "", "ta hedge_weights (M14.12) ne rend pas les bons poids sur ces losses : refais M14.12, ou "
                             "utilise ref_14.hedge_weights dans draw_hedge")
    for ax, eta, found in zip(axes, ETAS_14, founds_14):
        shown = f"{eta:g}".replace(".", ",")
        report_14(found == 5, f"η = {shown} : une courbe par expert, avec les bons poids",
                  f"η = {shown} : je trouve {found} expert(s) sur 5 dont les {len(LOSSES_14) + 1} poids (avant chaque "
                  f"tour, puis après le dernier) sont tracés ; trace chaque colonne de hedge_weights(losses, eta), une "
                  f"courbe ou une aire empilée par expert")
        title = ax.get_title()
        report_14(f"{eta:g}" in title or shown in title, f"η = {shown} : le titre donne η",
                  f"le titre de l'axe de η = {shown} doit donner la valeur de η (il vaut « {title} »)")
    plt.show()
    verdict("M14.14", all(results_14), "la figure est dans les axes reçus, avec les bons poids et un titre par axe.",
            "relis les lignes ❌ ci-dessus.")

## Partie C · Erreurs classiques, savoir-faire et défi

Fiche §1, §2 et §4. Quatre erreurs à trouver dans un résumé de calculs, un test de propriétés de la borne de Chernoff, puis Hedge sans connaître l'horizon.

### Ex M14.15 — Chasse à l'erreur : P(X ≥ k) calculée comme P(X > k), des covariances oubliées, des poids non renormalisés 🐛 ★ ⏱️ 15 min
**Objectif :** repérer dans un résumé de calculs les erreurs classiques d'une queue binomiale, d'une borne de Chernoff, d'une variance de moyenne et des poids de Hedge, puis calculer les bonnes valeurs.  
**Prérequis :** fiche §1, §2 et §4 · **Parcours :** C

Une collègue résume trois calculs (la cellule fournie) : la fiabilité d'une grappe de 16 serveurs qui fonctionne tant qu'au moins 13 sont en marche, chacun en marche avec la probabilité 0,93, indépendamment des autres ($X$, le nombre de serveurs en marche, suit la loi $\mathcal{B}(16 ;\ 0{,}93)$) ; la variance de la moyenne des erreurs de quatre modèles, de matrice de covariance `S_15` ; et les poids de Hedge de quatre experts, de losses cumulées `CUM_15`, avec $\eta = 0{,}35$. La fonction `kl_15(u, v)` rend $\mathrm{KL}(u \Vert v)$. Sept lignes portent une lettre et un commentaire qui dit ce qu'elles doivent calculer. Quatre lignes ne font pas ce que dit leur commentaire. **Sans exécuter son code**, trouve-les.

```python
n, k, p = N_15, 13, P_15                             # the cluster works while at least k = 13 servers are up
works = binom.sf(k, n, p)                                       # (A) P(X ≥ k): the probability that the cluster works
full = binom.sf(15 - 1, n, p)                                   # (B) P(X ≥ 15): at least 15 servers up
chernoff = np.exp(-n * kl_15(1 - p, A_15))                      # (C) the Chernoff bound on P(at least n·A_15 servers down)
var_plain = np.mean(np.diag(S_15)) / len(S_15)                  # (D) the variance of the plain mean of the 4 models
var_weighted = W_15 @ S_15 @ W_15                               # (E) the variance of their mean with the weights W_15
shifted = CUM_15 - CUM_15.min()                                 # (F) the cumulative losses minus the smallest one
weights = np.exp(-ETA_15 * shifted)                             # (G) the Hedge weights, a law on the 4 experts
```

Questions :
- a) les lettres des lignes fausses, dans l'ordre alphabétique (un texte) ;
- b) la bonne valeur de `works` (4 décimales) ;
- c) la bonne valeur de `chernoff` (4 décimales) ;
- d) la bonne valeur de `var_plain` (4 décimales) ;
- e) la bonne valeur de `weights` (une liste de 4 nombres, 4 décimales).

Pour b) à e), écris le code corrigé dans la cellule (c'est permis : seule la recherche des lignes fausses se fait sans exécuter).

In [ ]:
def kl_15(u, v):
    """KL(u ‖ v) between the Bernoulli laws of parameters u and v, in nats (0 < u, v < 1)."""
    return u * np.log(u / v) + (1 - u) * np.log((1 - u) / (1 - v))


N_15, P_15 = 16, 0.93            # 16 servers, each up with probability 0.93, independently of the others
A_15 = 0.25                      # an event: at least 16 × 0.25 = 4 servers down
S_15 = np.array([[1.5, 0.6, -0.3, 0.4],
                 [0.6, 2.2, 0.5, -0.2],
                 [-0.3, 0.5, 0.9, 0.1],
                 [0.4, -0.2, 0.1, 3.2]])     # the covariance matrix of the errors of 4 models
W_15 = np.array([0.3, 0.15, 0.4, 0.15])     # weights of sum 1
CUM_15 = np.array([12.5, 9.0, 10.5, 14.0])  # the cumulative losses of 4 experts
ETA_15 = 0.35

In [ ]:
answer_M14_15a = ...  # a) letters, as text
answer_M14_15b = ...  # b) a number (4 decimals)
answer_M14_15c = ...  # c) a number (4 decimals)
answer_M14_15d = ...  # d) a number (4 decimals)
answer_M14_15e = ...  # e) a list of 4 numbers (4 decimals)

In [ ]:
for letter, answer in zip("abcde", [answer_M14_15a, answer_M14_15b, answer_M14_15c, answer_M14_15d,
                                     answer_M14_15e]):
    wbm.check(f"M14.15{letter}", answer)

**Après la vérification**, dans ta copie : pour chaque ligne fausse, que calcule-t-elle exactement, et comment la corriger ? Laquelle sous-estime la fiabilité de la grappe, et laquelle donne un nombre plausible sans aucun signal ? Pourquoi les lignes (B) et (F), qui ressemblent à des lignes fausses, sont-elles justes ? La borne de (C) corrigée est-elle proche de la probabilité exacte qu'au moins 4 serveurs soient en panne, et pourquoi ?

### Ex M14.16 — Un test qui vérifie que la borne de Chernoff majore la queue exacte de `scipy.stats`, sur une grille de n, p et a 🛠️ ★ ⏱️ 15 min
**Objectif :** écrire un test de propriétés d'une borne, sur une grille qui contient les cas extrêmes, et vérifier qu'il attrape des bornes fausses.  
**Prérequis :** fiche §1 ; Ex M14.11 · **Parcours :** C

M11.26 vérifiait la seule propriété d'une borne de Markov : rester au-dessus de la queue. Une borne de Chernoff en a d'autres. Écris `test_chernoff_dominates(bound)`, une fonction de test dans le style de pytest (des `assert` avec un message qui dit le cas), qui reçoit une fonction `bound(n, p, a)` et vérifie, sur une grille de $n$, $p$ et $a$ de ton choix (au moins 100 cas, $a$ jusqu'à 1, des $p$ près de 0 et près de 1) :
- que `bound(n, p, a)` est dans $[0, 1]$ ;
- qu'elle majore la queue exacte $P(X \ge na)$ de la loi $\mathcal{B}(n, p)$, calculée par `binom.sf(k - 1, n, p)` avec $k = \lceil na \rceil$ (attention aux flottants : `50 * 0.14` vaut `7.000000000000001`) ;
- que, pour $a > p$, elle reste sous la borne de Hoeffding $e^{-2n(a - p)^2}$ (fiche §1 : l'inégalité de Pinsker) ;
- que, pour $a > p$, elle ne croît pas avec $n$ (compare par exemple `bound(n, p, a)` et `bound(2 * n, p, a)`).

Prévois une petite tolérance (`1e-12`) dans chaque comparaison. La vérification appelle ton test sur la vraie borne, `ref_16.chernoff_bound`, où il doit **passer**, et compte les cas de ta grille ; puis sur les quatre fausses bornes de la cellule fournie, où il doit **échouer** : `bound_reversed`, $e^{-n\,\mathrm{KL}(p \Vert a)}$ ; `bound_squared`, $e^{-2n\,\mathrm{KL}(a \Vert p)}$ ; `bound_fixed_lambda`, l'inégalité de Markov appliquée à $e^{\lambda X}$ avec $\lambda = 1$ au lieu du meilleur $\lambda$ ; `bound_markov`, la borne de Markov $\min(1, p/a)$.

Dans ta copie : quelle propriété attrape chacune des quatre fausses bornes, et sur quel cas de ta grille ? Deux d'entre elles sont de **vraies** bornes de la queue, toujours au-dessus d'elle : lesquelles, et pourquoi ton test les refuse-t-il quand même ? Pourquoi `bound_fixed_lambda` peut-elle rendre plus que 1 ?

In [ ]:
ref_16 = wbm.load_mymath("ref", alias="mymath_ref").ensembles


def bound_reversed(n, p, a):
    """A wrong bound: exp(-n KL(p ‖ a)), the divergence the wrong way round (1 for a ≤ p, 0 for a = 1)."""
    if a <= p:
        return 1.0
    if a >= 1.0:
        return 0.0
    return math.exp(-n * (p * math.log(p / a) + (1 - p) * math.log((1 - p) / (1 - a))))


def bound_squared(n, p, a):
    """A wrong bound: exp(-2 n KL(a ‖ p)), the square of the Chernoff bound."""
    return ref_16.chernoff_bound(n, p, a) ** 2


def bound_fixed_lambda(n, p, a):
    """Markov's inequality on e^(λX) with λ = 1 instead of the best λ: e^(-n a) (1 - p + p e)^n (capped at e^700)."""
    return math.exp(min(700.0, n * (math.log(1 - p + p * math.e) - a)))


def bound_markov(n, p, a):
    """Markov's bound min(1, p / a) on P(X ≥ n a) (1 for a = 0)."""
    return min(1.0, p / a) if a > 0 else 1.0


WRONG_16 = {"bound_reversed (e^(−n KL(p ‖ a)))": bound_reversed,
            "bound_squared (e^(−2n KL))": bound_squared,
            "bound_fixed_lambda (Markov sur e^X, λ = 1)": bound_fixed_lambda,
            "bound_markov (min(1, p/a))": bound_markov}

In [ ]:
def test_chernoff_dominates(bound):
    """bound(n, p, a) is in [0, 1], above the exact tail P(X >= n a) of B(n, p), below Hoeffding's bound for a > p,
    and does not increase with n for a > p: on a grid of n, p and a."""
    raise NotImplementedError("test_chernoff_dominates n'est pas encore écrit (04_indices.md si tu bloques)")

In [ ]:
with checked("M14.16"):
    if len(inspect.signature(test_chernoff_dominates).parameters) != 1:
        raise TypeError("ton test prend un seul argument, la borne à tester : def test_chernoff_dominates(bound)")
    results_16, calls_16 = [], []

    def report_16(ok, success, failure):
        results_16.append(bool(ok))
        print(("✅ " + success) if ok else ("❌ " + failure))

    def recorded_16(n, p, a):
        """The true bound, which records the cases of your grid."""
        calls_16.append((n, float(p), float(a)))
        return ref_16.chernoff_bound(n, p, a)

    try:
        test_chernoff_dominates(recorded_16)          # your test on the true bound: ⏳ while it is not written
        why_16 = ""
    except AssertionError as exc:
        why_16 = str(exc)[:200] or "un assert sans message"
    report_16(not why_16, "ton test passe sur la vraie borne de Chernoff",
              f"ton test échoue sur la vraie borne de Chernoff : il refuse une borne juste ({why_16})")
    cases_16 = set(calls_16)
    report_16(len(cases_16) >= 100, f"ta grille compte {len(cases_16)} cas différents",
              f"ta grille ne compte que {len(cases_16)} cas différents : il en faut au moins 100")
    report_16(bool(cases_16) and max(a for _, _, a in cases_16) >= 1 - 1e-9, "a va jusqu'à 1",
              "ta grille doit aller jusqu'à a = 1 (le cas où 0 ln 0 compte pour 0)")
    report_16(bool(cases_16) and min(p for _, p, _ in cases_16) <= 0.05 and max(p for _, p, _ in cases_16) >= 0.95,
              "p va près de 0 et près de 1",
              "ta grille doit contenir des p près de 0 (au plus 0,05) et près de 1 (au moins 0,95) : les événements rares")
    hints_16 = {"bound_reversed": "compare-la à la queue exacte, aussi pour a = 1 et de petits n",
                "bound_squared": "compare-la à la queue exacte, aussi pour de petits n",
                "bound_fixed_lambda": "vérifie que la borne reste dans [0, 1], aussi pour a ≤ p",
                "bound_markov": "compare-la à la borne de Hoeffding, pour a > p et de grands n"}
    for label_16, wrong_16 in WRONG_16.items():
        try:
            test_chernoff_dominates(wrong_16)
            report_16(False, "", f"ton test passe sur {label_16} : il ne la voit pas ; "
                                 f"{hints_16[wrong_16.__name__]}")
        except AssertionError:
            report_16(True, f"ton test échoue sur {label_16}", "")
        except NotImplementedError:
            raise
        except Exception as exc:  # noqa: BLE001 - a crash is not a failed assert
            report_16(False, "", f"ton test plante sur {label_16} ({type(exc).__name__} : {str(exc)[:100]}) au lieu "
                                 "d'échouer sur un assert")
    verdict("M14.16", all(results_16), "ton test accepte la vraie borne, couvre la grille demandée et attrape les "
                                       "quatre fausses bornes.", "relis les lignes ❌ ci-dessus.")

### Ex M14.17 — Défi : suivre le meilleur de dix experts avec Hedge sans connaître l'horizon, un regret sous la borne théorique sur 1 000 tours 🏆 ★★★ ⏱️ 25 min
**Objectif :** régler le learning rate de Hedge sans connaître l'horizon, pour garder un regret sous la borne théorique à tout moment, sur des suites de losses variées.  
**Prérequis :** fiche §4 ; Ex M14.12 et M14.14 · **Parcours :** C

Hedge avec $\eta = \sqrt{8\ln K/T}$ demande l'horizon $T$. Ici, tu ne le connais pas : les tours arrivent un à un, et ton regret est mesuré **à tout moment**, $R_t = \sum_{s \le t} \mathbf{w}_s \cdot \boldsymbol{\ell}_s - \min_k L_{t+1}[k]$.

**L'objectif** : écrire une classe `AnytimeHedge(K)`, pour $K$ experts, avec deux méthodes :
- `weights()` rend ta loi actuelle sur les experts : un tableau de $K$ nombres positifs ou nuls, de somme 1 ;
- `update(losses)` reçoit les losses du tour, un vecteur de $K$ nombres dans $[0, 1]$.

La vérification joue 1 000 tours sur trois suites de losses de 10 experts (cellule fournie, graines fixes) : `"stochastique"`, des losses aléatoires avec un meilleur expert net ; `"changement"`, où le meilleur expert change au 501ᵉ tour ; `"piège"`, où deux experts alternent en tête, ce qui fait échouer « suivre le meneur » (tout le poids sur l'expert de plus petite loss cumulée, $\eta = +\infty$). À chaque tour, elle lit `weights()`, ajoute $\mathbf{w}_t \cdot \boldsymbol{\ell}_t$ à ta loss, puis appelle `update(losses)`. Aux tours 50, 200 et 1 000, elle compare ton regret à la borne $\sqrt{2t\ln K} + \sqrt{\frac{\ln K}{8}}$ (fiche §4) : il doit être dessous à chacun des neuf points de contrôle. `hedge_weights`, `mymath`, `scipy`, `sklearn`, `sequence_17`, `SEQUENCES_17`, `T_17` et `CHECKPOINTS_17` sont interdits dans ta classe.

Essaie d'abord un $\eta$ constant (0,01, puis le meilleur pour $T = 1\,000$) et « suivre le meneur », pour voir où chacun échoue. Les indices de `04_indices.md` donnent des pistes, à ouvrir une à une.

La cellule de vérification affiche, pour chaque suite, ton regret à chaque point de contrôle (✅ ou ❌), puis enregistre si l'objectif est atteint.

Dans ta copie : où échouent $\eta = 0{,}01$, le meilleur $\eta$ constant pour $T = 1\,000$ et « suivre le meneur », et pourquoi ? Qu'as-tu choisi, et pourquoi ta méthode ne dépend-elle pas de $T$ ? Que vaut ton regret au tour 1 000 sur chaque suite, comparé à la borne ?

In [ ]:
K_17, T_17, CHECKPOINTS_17 = 10, 1000, (50, 200, 1000)
SEQUENCES_17 = {"stochastique": 4337, "changement": 4338, "piège": 4339}     # the kind of each sequence, and its seed
ref_17 = wbm.load_mymath("ref", alias="mymath_ref").ensembles


def sequence_17(kind, seed):
    """The (T_17, K_17) losses, in [0, 1], of a sequence of the check."""
    rng = np.random.default_rng(seed)
    if kind == "stochastique":                       # 0/1 losses; expert 4 loses far less often than the others
        means = rng.uniform(0.6, 0.9, size=K_17)
        means[3] = 0.05
        return (rng.random((T_17, K_17)) < means).astype(float)
    if kind == "changement":                         # expert 2 is the best for 500 rounds, then expert 9
        means = rng.uniform(0.45, 0.75, size=(2, K_17))
        means[0, 1] = means[1, 8] = 0.2
        return np.clip(np.repeat(means, T_17 // 2, axis=0) + rng.normal(0.0, 0.15, size=(T_17, K_17)), 0.0, 1.0)
    losses = np.clip(0.75 + rng.normal(0.0, 0.1, size=(T_17, K_17)), 0.0, 1.0)   # "piège": experts 1 and 2 alternate
    rounds = np.arange(T_17)
    losses[:, 0] = np.where(rounds % 2 == 1, 0.0, 1.0)
    losses[:, 1] = np.where(rounds % 2 == 1, 1.0, 0.0)
    losses[0, :2] = (0.5, 0.0)
    return losses


def regret_bound_17(t):
    """The bound √(2t ln K) + √(ln K / 8) on the regret after t rounds, for K = K_17 experts."""
    return math.sqrt(2 * t * math.log(K_17)) + math.sqrt(math.log(K_17) / 8)

In [ ]:
class AnytimeHedge:
    """Hedge on K experts without knowing the horizon: weights() is the current law on the experts (K non-negative
    numbers of sum 1), update(losses) adds the K losses of the round (numbers in [0, 1])."""

    def __init__(self, K):
        raise NotImplementedError("AnytimeHedge n'est pas encore écrite (04_indices.md si tu bloques)")

    def weights(self):
        raise NotImplementedError("AnytimeHedge n'est pas encore écrite (04_indices.md si tu bloques)")

    def update(self, losses):
        raise NotImplementedError("AnytimeHedge n'est pas encore écrite (04_indices.md si tu bloques)")

In [ ]:
with checked("M14.17 défi"):
    problems_17, ok_17, ran_17 = [], False, False
    sources_17 = []
    for item in vars(AnytimeHedge).values():     # the class block, read back from the cell that defines its methods
        filename = getattr(getattr(item, "__code__", None), "co_filename", None)
        if filename and not sources_17:
            lines = linecache.getlines(filename)
            try:
                sources_17 += [textwrap.dedent("".join(lines[node.lineno - 1:node.end_lineno]))
                               for node in ast.walk(ast.parse("".join(lines)))
                               if isinstance(node, ast.ClassDef) and node.name == "AnytimeHedge"]
            except SyntaxError:
                pass
    for item in [AnytimeHedge, *vars(AnytimeHedge).values()]:   # and each method on its own
        try:
            sources_17.append(textwrap.dedent(inspect.getsource(item)))
        except (OSError, TypeError):
            pass
    code_17 = ""
    for source in sources_17:                        # the code only: comments and strings (docstrings) may name anything
        try:
            code_17 += " " + " ".join(tok.string for tok in tokenize.generate_tokens(io.StringIO(source).readline)
                                      if tok.type not in (tokenize.COMMENT, tokenize.STRING))
        except (tokenize.TokenError, SyntaxError):
            code_17 += " " + source
    banned_17 = sorted(set(re.findall(r"\b(hedge_weights|mymath|mymath_ref|ref_17|scipy|sklearn|sequence_17|"
                                      r"SEQUENCES_17|T_17|CHECKPOINTS_17)\b", code_17)))
    if banned_17:
        ran_17 = True
        problems_17.append(f"AnytimeHedge utilise {', '.join(banned_17)}, interdit dans ce défi")
    start_17 = time.perf_counter()
    for name_t, seed_t in ([] if banned_17 else SEQUENCES_17.items()):   # the three sequences, each with a new agent
        losses_t = sequence_17(name_t, seed_t)
        agent_t = AnytimeHedge(K_17)                 # ⏳ while the class is not written
        total_t, cumulative_t, regrets_t, broken_t = 0.0, np.zeros(K_17), {}, ""
        try:
            with own_code("M14.17", "AnytimeHedge", {"hedge_weights": ref_17.hedge_weights}):
                for t in range(T_17):
                    w_t = np.asarray(agent_t.weights(), dtype=float).reshape(-1)
                    if (w_t.shape != (K_17,) or not np.all(np.isfinite(w_t)) or w_t.min() < -1e-12
                            or abs(w_t.sum() - 1.0) > 1e-9):
                        broken_t = (f"au tour {t + 1}, weights() doit rendre {K_17} nombres positifs ou nuls de somme 1, "
                                    f"reçu {np.round(w_t, 4).tolist()[:12]}")
                        break
                    total_t += float(w_t @ losses_t[t])
                    cumulative_t += losses_t[t]
                    agent_t.update(losses_t[t].copy())
                    if t + 1 in CHECKPOINTS_17:
                        regrets_t[t + 1] = total_t - cumulative_t.min()
        except NotImplementedError:
            raise
        except StopCheck:                             # own_code has said which reference function was called
            ran_17 = True
            problems_17.append("AnytimeHedge appelle une fonction de la référence")
            break
        except Exception as exc:  # noqa: BLE001
            broken_t = f"AnytimeHedge lève {type(exc).__name__} ({exc})"
        ran_17 = True
        marks_t = " ; ".join(f"au tour {t} : {fr(r, 2)} (borne {fr(regret_bound_17(t), 2)}) "
                             + ("✅" if r <= regret_bound_17(t) else "❌") for t, r in regrets_t.items())
        if broken_t:
            print(f"❌ « {name_t} » : {broken_t}" + (f" ; avant cela, regret {marks_t}" if marks_t else ""))
            problems_17.append(f"« {name_t} » : {broken_t}")
            continue
        late_17 = [str(t) for t, r in regrets_t.items() if not r <= regret_bound_17(t)]
        print(("❌" if late_17 else "✅") + f" « {name_t} » : regret {marks_t}")
        if late_17:
            problems_17.append(f"« {name_t} » : regret au-dessus de la borne au(x) tour(s) {', '.join(late_17)}")
    if time.perf_counter() - start_17 > 60:
        problems_17.append(f"{time.perf_counter() - start_17:.0f} s pour les trois suites, plus que les 60 s permises")
    if problems_17:
        print("❌ Ex M14.17 : objectif pas encore atteint : " + " ; ".join(problems_17) + ".")
    elif ran_17:
        ok_17 = True
        print("✅ Ex M14.17 : défi réussi : un regret sous la borne aux tours 50, 200 et 1 000, sur les trois suites, "
              "sans connaître l'horizon.")
with checked("M14.17"):
    if not ran_17:                               # ⏳ while AnytimeHedge is not written
        raise NotImplementedError("AnytimeHedge n'est pas encore écrite")
    wbm.check("M14.17", bool(ok_17), computed=True, quiet=not ok_17)   # the lines above say what fails

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu calculer la queue d'une loi binomiale, et la majorer par les bornes de Markov, de Hoeffding et de Chernoff, en sachant laquelle est la meilleure ?
2. Sais-tu calculer la variance d'une moyenne pondérée par $\mathbf{w}^\top\boldsymbol{\Sigma}\mathbf{w}$, et expliquer le plancher que laissent les covariances ?
3. Sais-tu utiliser $(1 + x/n)^n \to e^x$ et estimer son erreur ?
4. Sais-tu calculer les poids de Hedge et son regret, et régler son learning rate, même sans connaître l'horizon ?
5. Sais-tu calculer le gradient d'une loss par rapport au vecteur des prédictions, et lire les poids qu'il donne aux exemples ?

**La suite** : le chapitre 14 de workbookIA (des ensembles de modèles : vote, bagging, forêts aléatoires, AdaBoost et gradient boosting), puis CP3, l'examen blanc de la partie III, juste avant le checkpoint III de workbookIA.